In [1]:
from googleapiclient.discovery import build
import csv
from datetime import datetime, timezone
import time

In [ ]:
# 설정
API_KEY = ""
OUTPUT_FILE = "youtube_public_playlists_full_last2.csv"
MAX_COMMENTS_PER_VIDEO = 500

# 수집 대상 공개 채널 (채널 ID)
CHANNELS = {
    "": ""
}

# YouTube API 빌드
youtube = build("youtube", "v3", developerKey=API_KEY)

In [11]:
# 공개 채널 재생목록 가져오기
def fetch_playlists(channel_id):
    playlists = []
    next_page_token = None
    while True:
        res = youtube.playlists().list(
            part="id,snippet",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for pl in res.get("items", []):
            playlists.append({
                "playlist_id": pl["id"],
                "title": pl["snippet"]["title"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return playlists

In [12]:
# 재생목록 안 영상 가져오기
def fetch_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    while True:
        res = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for item in res.get("items", []):
            videos.append({
                "video_id": item["snippet"]["resourceId"]["videoId"],
                "title": item["snippet"]["title"],
                "description": item["snippet"]["description"],
                "publish_time": item["snippet"]["publishedAt"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return videos

In [13]:
# 영상 통계 조회
def fetch_video_stats(video_id):
    resp = youtube.videos().list(
        part="snippet,statistics",
        id=video_id
    ).execute()

    if not resp["items"]:
        return None

    item = resp["items"][0]

    return {
        "upload_time": item["snippet"]["publishedAt"], # :point_left: 이게 진짜 업로드 시각
        "viewCount": item["statistics"].get("viewCount","0"),
        "likeCount": item["statistics"].get("likeCount","0"),
        "commentCount": item["statistics"].get("commentCount","0")
    }

In [14]:
# 댓글 수집 + 좋아요
def fetch_comments(video_id):
    all_comments = []
    next_page_token = None

    while True:
        res = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            pageToken=next_page_token
        ).execute()

        for thread in res.get("items", []):
            top = thread["snippet"]["topLevelComment"]["snippet"]
            all_comments.append({
                "text": top["textDisplay"],
                "likeCount": top.get("likeCount", 0)
            })

            if "replies" in thread:
                for reply in thread["replies"]["comments"]:
                    r = reply["snippet"]
                    all_comments.append({
                        "text": r["textDisplay"],
                        "likeCount": r.get("likeCount", 0)
                    })

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

    # 좋아요 내림차순 정렬 후 최대 500개만
    all_comments.sort(key=lambda x: x["likeCount"], reverse=True)
    return all_comments[:500]

In [15]:
# 영상 + 댓글 + 통계 처리
def process_video(video, playlist_title):
    video_id = video["video_id"]
    publish_utc = datetime.fromisoformat(video["publish_time"].replace("Z","+00:00")).strftime("%Y-%m-%d %H:%M:%S")

    stats = fetch_video_stats(video_id)
    comments = fetch_comments(video_id)

    rows = []
    if not comments:
        rows.append([
            playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
            stats["viewCount"], stats["likeCount"], stats["commentCount"],
            "", ""])
    else:
        for c in comments:
            rows.append([
                playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
                stats["viewCount"], stats["likeCount"], stats["commentCount"],
                c["text"], c["likeCount"]
            ])
    return rows

In [16]:
# 메인 수집
def collect_youtube_public_playlists():
    total_processed = 0
    start_time = time.time()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "playlist_title",
            "video_id",
            "title",
            "description",
            "publish_time_utc",
            "viewCount",
            "likeCount",
            "commentCount",
            "comment_text",
            "comment_likeCount"
        ])

        for channel_name, channel_id in CHANNELS.items():
            print(f"\n===== 채널 수집 시작: {channel_name} =====")
            playlists = fetch_playlists(channel_id)
            print(f"총 {len(playlists)}개의 재생목록 발견")

            for pl in playlists:
                playlist_title = pl["title"]
                print(f"\n----- 재생목록 수집 시작: {playlist_title} -----")
                videos = fetch_playlist_videos(pl["playlist_id"])
                print(f"총 {len(videos)}개의 영상 발견")

                for video in videos:
                    rows = process_video(video, playlist_title)
                    for row in rows:
                        writer.writerow(row)
                    total_processed += 1

                    # 진행률 및 ETA
                    progress_ratio = total_processed / max(len(videos),1)
                    elapsed_time = time.time() - start_time
                    avg_time = elapsed_time / max(total_processed,1)
                    remaining_time = avg_time * (len(videos) - total_processed)

                    if total_processed % 5 == 0:
                        print(
                            f"[{playlist_title}] 저장 중 영상 ID: {rows[0][1]} | "
                            f"게시글 UTC: {rows[0][4]} | "
                            f"진행률: {progress_ratio*100:.2f}% | "
                            f"총 처리: {total_processed}개 | "
                            f"평균: {avg_time:.2f}초/개 | "
                            f"예상 남은 시간: {remaining_time/60:.1f}분"
                        )

                print(f"----- 재생목록 수집 완료: {playlist_title} -----")

            print(f"===== 채널 수집 완료: {channel_name} =====")

    print("\n===== 전체 YouTube 공개 재생목록 수집 완료 =====")


In [17]:
# 실행
collect_youtube_public_playlists()


===== 채널 수집 시작: 재열 =====
총 2개의 재생목록 발견

----- 재생목록 수집 시작: 롤 패치노트 -----
총 28개의 영상 발견
[롤 패치노트] 저장 중 영상 ID: hx2DbjzWQ5I | 게시글 UTC: 2025-02-27T12:30:44Z | 진행률: 17.86% | 총 처리: 5개 | 평균: 1.61초/개 | 예상 남은 시간: 0.6분
[롤 패치노트] 저장 중 영상 ID: J8DXjj71c38 | 게시글 UTC: 2025-05-08T11:30:13Z | 진행률: 35.71% | 총 처리: 10개 | 평균: 1.34초/개 | 예상 남은 시간: 0.4분
[롤 패치노트] 저장 중 영상 ID: oF0IrPupo_I | 게시글 UTC: 2025-07-24T11:30:43Z | 진행률: 53.57% | 총 처리: 15개 | 평균: 1.20초/개 | 예상 남은 시간: 0.3분
[롤 패치노트] 저장 중 영상 ID: 6A78gkuagpk | 게시글 UTC: 2025-10-02T11:30:07Z | 진행률: 71.43% | 총 처리: 20개 | 평균: 1.11초/개 | 예상 남은 시간: 0.1분
[롤 패치노트] 저장 중 영상 ID: v1tcDbKDkp0 | 게시글 UTC: 2026-01-06T12:31:37Z | 진행률: 89.29% | 총 처리: 25개 | 평균: 1.12초/개 | 예상 남은 시간: 0.1분
----- 재생목록 수집 완료: 롤 패치노트 -----

----- 재생목록 수집 시작: 롤 개발자노트 -----
총 8개의 영상 발견
[롤 개발자노트] 저장 중 영상 ID: 3zx1PwFXFdQ | 게시글 UTC: 2024-11-25T16:00:03Z | 진행률: 375.00% | 총 처리: 30개 | 평균: 1.76초/개 | 예상 남은 시간: -0.6분
[롤 개발자노트] 저장 중 영상 ID: R5Ff-0aUpIk | 게시글 UTC: 2025-12-01T16:00:47Z | 진행률: 437.50% | 총 처리: 35개 | 평균: 4.43초/